# Create Temporary Tables using BigQuery - bikesharing in Austin, Texas Dataset

## Activity overview

In this notebook, we will make one temporary table using BigQuery and use it to run a query.

## Objective
Use temporary tables to work with data without changing the original data

## What are temp tables?

As data calculations become more complicated, there are many components to keep track of. This is similar to keeping track of tasks in daily life. Some people use sticky notes while others use checklists. In data science, a temporary table is just like a sticky note.

Temporary tables, or temp tables, store subsets of data from standard data tables for a certain period of time. When we end our SQL database session, they are automatically deleted. Temp tables allow us to run calculations in temporary data tables without needing to make modifications to the primary tables in our database.

Now, we will create a temp table.

## Importing data

To begin, import our data. We will use a dataset on <b> bikesharing in Austin, Texas</b>. Specifically, we will work with a table that gives details about each public bike ride’s duration, starting station, and ending station.

In [1]:
from google.cloud import bigquery

print(bigquery.__version__)

3.40.1


In [2]:
client = bigquery.Client()

print(client.project)

myproject001-504709


## Create a temporary table

Let's consider the following scenario: A bikeshare company has reached a recent milestone, and their marketing team wants to write a blog post that “congratulates” their most-used bike on being so popular. They want to include the name of the station that the bike is most likely to be found. 

They task us with figuring out the station from which the bike begins a trip most frequently. 

In order to do this, we will need to create a temp table to find the ID number of the bike that has taken the longest total trips (in minutes). We will take a sum of the minutes of each trip for each bike, then sort by descending order to find the bike that has spent the most minutes being used.

To do that, let's follow the steps below:

1. Begin our query with WITH to set up a temp table. Name it longest_used_bike.

2. Create a subquery after AS

5. SELECT bikeid and create SUM(duration_minutes) AS trip_duration. This creates a column in the temp table that contains the sum of the total minutes a bike has been used.

6. GROUP BY bikeid.

7. ORDER BY trip_duration DESC

This sets up our temporary table. This section identifies the specific bike (bikeid) with the longest trip duration.

### Write our query

Now that we have found the ID of the bike that has been used the longest, we will write a query to find the station from which this bike leaves most frequently. To do this, we will join your temp table (containing just the bike’s ID) with the original table and return the station ID with the highest number of trips started.

In [4]:
query = """
WITH
    longest_used_bike AS (
        SELECT
            bike_id,
            SUM(duration_minutes) AS trip_duration
        FROM
            bigquery-public-data.austin_bikeshare.bikeshare_trips
        GROUP BY
            bike_id
        ORDER BY
            trip_duration DESC
        LIMIT 1
    )

#find station at which longest bikeshare ride started
SELECT
    trips.start_station_id,
    COUNT(*) AS trip_ct                #count how many times the bike has left each station
FROM
    longest_used_bike AS longest
INNER JOIN                             #to pick out the station ID that corresponds to the bike we identified in the temporary table.
    `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS trips
ON longest.bike_id = trips.bike_id
GROUP BY
    trips.start_station_id
ORDER BY
    trip_ct DESC
LIMIT 1
"""
df = client.query(query).to_dataframe()
df

,start_station_id,trip_ct
0,3798,177
